In [5]:
import pandas as pd
import geopandas as gpd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import calendar
import shapely as wkt
import ast
import re

import folium  # For plots of geometry on interactive map
import matplotlib.colors as mcolors
import random

import plotly.express as px
import plotly.io as pio

pio.renderers.default = "colab"  # or: "notebook_connected"
from plotly.graph_objs import Font

import sys
from pathlib import Path

DATA_DIR = Path.cwd().parent / "data"
print("Data directory:", DATA_DIR.resolve())

Data directory: C:\Users\cort3\Documents\Classes\OptimalChargerPlacement\data


## EV Projection Data for Census Blocks

In [23]:
# Import the EV Projection Dataframe
# Choose which years to import and get growth
# Choose if including all, public, or private chargers
# Choose managed vs unmanaged chargers


def import_ev_projection_df(year, type="unmanaged"):
    if type not in ["managed", "unmanaged", "all"]:
        raise ValueError("type must be 'managed', 'unmanaged', or 'all'")
    if year not in [ "full_ev_adoption_alameda_load_curves.csv", "2035", "2025"]:
        raise ValueError("year must be 'full_ev_adoption_alameda_load_curves.csv', '2035', or '2025'")

    if type == "unmanaged":
        if year == "full_addoption":
            rel_file_path = "full_ev_adoption_alameda_load_curves.csv"
        elif year == "2035":
            rel_file_path = "sim_20251117_alameda_2035_adoption/ev_load_curves.csv"
        elif year == "2025":
            rel_file_path = "sim_20251117_alameda_2025_adoption/ev_load_curves.csv"

    elif type == "managed":
        if year == "full_addoption":
            rel_file_path = "full_ev_adoption_alameda_load_curves.csv"
        elif year == "2035":
            rel_file_path = "sim_20251121_alameda_2035_managed_naive/ev_load_curves.csv"
        elif year == "2025":
            rel_file_path = "sim_20251121_alameda_2025_managed_naive/ev_load_curves.csv"

    print(year)
    file_path = os.path.join(DATA_DIR, "EvChargingLoadCurves", rel_file_path)
    df = pd.read_csv(file_path)
    # Convert string "[...]" → Python list of floats
    if "load_curve" in df.columns:
        df["load_curve"] = df["load_curve"].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else x
        )
    return df


ev_projection_df = import_ev_projection_df("2035", type="managed")
display(ev_projection_df.head())

2035


,geoid,charger_type,number_of_sessions,load_curve
0,"0 (Tract 9900, Alameda, CA)",Public_HD_Long_Duration_DCFC_150_kW,4,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,"0 (Tract 9900, Alameda, CA)",Public_HD_Long_Duration_DCFC_350_kW,4,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,"0 (Tract 9900, Alameda, CA)",Public_MD_Long_Duration_DCFC_150_kW,2,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,"0 (Tract 9900, Alameda, CA)",Public_MD_Long_Duration_DCFC_50_kW,3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,"0 (Tract 9900, Alameda, CA)",Public_MD_Long_Duration_L2_11_kW,9,"[11.2, 11.2, 11.2, 11.2, 11.2, 11.2, 11.2, 11...."


In [22]:

def expand_load_curve_to_hourly(df, method="mean"):
    """
    Expands 96-point 15-minute load curves into 24 hourly columns.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain a column named 'load_curve' with lists of length 96.
    method : str
        "mean" = average of 4 quarter-hour values
        "max"  = max of 4 quarter-hour values

    Returns: Original df with 24 new columns: hour_0 ... hour_23
    """

    if method not in ["mean", "max"]:
        raise ValueError("method must be 'mean' or 'max'")

    # Function to aggregate one row’s curve into 24 hourly values
    def aggregate_curve(curve):
        curve = np.array(curve)
        hourly = []
        for h in range(24):
            block = curve[h * 4 : (h + 1) * 4]
            if method == "mean":
                hourly.append(block.mean())
            else:  # method == "max"
                hourly.append(block.max())
        return hourly

    # Expand into a DataFrame
    hourly_expanded = df["load_curve"].apply(aggregate_curve)
    hourly_df = pd.DataFrame(
        hourly_expanded.tolist(), columns=[f"{i}" for i in range(24)]
    )

    # Return original df + new columns
    return pd.concat([df.drop(columns=["load_curve"]), hourly_df], axis=1)

hrly = expand_load_curve_to_hourly(ev_projection_df, method="mean")
display(hrly.head())
# list all unique values in charger type column
print(ev_projection_df["charger_type"].unique())

,geoid,charger_type,number_of_sessions,0,1,2,3,4,5,6,...,14,15,16,17,18,19,20,21,22,23
0,"0 (Tract 9900, Alameda, CA)",Public_HD_Long_Duration_DCFC_150_kW,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,367.500000,309.75,15.75,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
1,"0 (Tract 9900, Alameda, CA)",Public_HD_Long_Duration_DCFC_350_kW,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,469.000000,162.75,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
2,"0 (Tract 9900, Alameda, CA)",Public_MD_Long_Duration_DCFC_150_kW,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
3,"0 (Tract 9900, Alameda, CA)",Public_MD_Long_Duration_DCFC_50_kW,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,52.500000,35.00,35.00,35.0,35.0,35.0,35.0,35.0,35.0,14.639984
4,"0 (Tract 9900, Alameda, CA)",Public_MD_Long_Duration_L2_11_kW,9,11.2,11.2,11.2,11.2,11.2,11.2,11.2,...,22.400001,11.20,11.20,11.2,11.2,11.2,11.2,11.2,11.2,11.200000


['Public_HD_Long_Duration_DCFC_150_kW'
 'Public_HD_Long_Duration_DCFC_350_kW'
 'Public_MD_Long_Duration_DCFC_150_kW'
 'Public_MD_Long_Duration_DCFC_50_kW' 'Public_MD_Long_Duration_L2_11_kW'
 'Public_MD_Opportunistic_DCFC_350_kW'
 'Public_MD_Opportunistic_DCFC_50_kW' 'Public_LD_L2_11_KW' 'Work_LD_L2'
 'Public_LD_DCFC_150_kW' 'SFH_LD_L2' 'MFH_LD_L2' 'Public_LD_DCFC_250_kW'
 'Public_LD_DCFC_350_kW' 'Public_HD_Long_Duration_DCFC_50_kW'
 'Public_HD_Long_Duration_DCFC_750_kW'
 'Public_HD_Opportunistic_DCFC_350_kW'
 'Public_MD_Opportunistic_DCFC_1000_kW'
 'Public_HD_Opportunistic_DCFC_50_kW'
 'Public_HD_Opportunistic_DCFC_1000_kW']


In [ ]:
def sum_ev_loads_by_charger_type(df, charger_type="all"):
    public_types = [
        "Public_HD_Long_Duration_DCFC_150_kW",
        "Public_HD_Long_Duration_DCFC_350_kW",
        "Public_MD_Long_Duration_DCFC_150_kW",
        "Public_MD_Long_Duration_DCFC_50_kW",
        "Public_MD_Long_Duration_L2_11_kW",
        "Public_MD_Opportunistic_DCFC_350_kW",
        "Public_MD_Opportunistic_DCFC_50_kW",
        "Public_LD_L2_11_KW",
        "Public_LD_DCFC_150_kW",
        "Public_LD_DCFC_250_kW",
        "Public_LD_DCFC_350_kW",
        "Public_HD_Long_Duration_DCFC_50_kW",
        "Public_HD_Long_Duration_DCFC_750_kW",
        "Public_HD_Opportunistic_DCFC_350_kW",
        "Public_MD_Opportunistic_DCFC_1000_kW",
        "Public_HD_Opportunistic_DCFC_50_kW",
        "Public_HD_Opportunistic_DCFC_1000_kW",
    ]

    private_types = ["SFH_LD_L2", "Work_LD_L2", "MFH_LD_L2"]

    if charger_type == "all":
        filtered_df = df.copy()
    elif charger_type == "public":
        filtered_df = df[df["charger_type"].isin(public_types)].copy()
    else:  # private
        filtered_df = df[df["charger_type"].isin(private_types)].copy()

    hourly_cols = [str(i) for i in range(24)]

    # --- CORE FIX: ensure every geoid appears ---
    all_geoids = df["geoid"].unique()

    summed_df = (
        filtered_df.groupby("geoid")[hourly_cols]
        .sum()
        .reindex(all_geoids, fill_value=0)  # <--- THIS LINE FIXES THE PROBLEM
        .reset_index()
    )

    return summed_df
# summed_loads_public = sum_ev_loads_by_charger_type(hrly, charger_type="public")
# display(summed_loads_public.head())
# summed_loads_private = sum_ev_loads_by_charger_type(hrly, charger_type="private")
# display(summed_loads_private.head())
# summed_loads_all = sum_ev_loads_by_charger_type(hrly, charger_type="all")
# display(summed_loads_all.head())

,geoid,0,1,2,3,4,5,6,7,8,...,14,15,16,17,18,19,20,21,22,23
0,"0 (Tract 9900, Alameda, CA)",11.2,11.2,11.2,11.2,11.2,11.200000,11.2,2.8,0.00,...,911.400001,518.7,61.950000,46.20,46.200,46.2,46.2,46.2,46.2,25.839984
1,"0 (Tract 9901, San Mateo, CA)",0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,236.25,...,0.000000,0.0,0.000000,0.00,0.000,0.0,0.0,0.0,0.0,0.000000
2,"1 (Tract 1, Carson City, NV)",22.4,22.4,22.4,22.4,22.4,10.960008,0.0,0.0,0.00,...,0.000000,0.0,0.000000,0.00,0.000,0.0,0.0,5.6,22.4,22.400000
3,"1 (Tract 1, Fresno, CA)",11.2,11.2,16.8,22.4,22.4,22.400000,22.4,22.4,22.40,...,11.200000,11.2,4.400006,0.00,0.000,0.0,0.0,0.0,0.0,11.200000
4,"1 (Tract 1, Inyo, CA)",0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.00,...,0.000000,0.0,0.000000,68.25,34.125,0.0,0.0,0.0,0.0,0.000000


,geoid,0,1,2,3,4,5,6,7,8,...,14,15,16,17,18,19,20,21,22,23
0,"0 (Tract 9900, Alameda, CA)",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,"0 (Tract 9901, San Mateo, CA)",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,"1 (Tract 1, Carson City, NV)",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,"1 (Tract 1, Fresno, CA)",7.2,7.2,7.2,7.2,7.2,7.2,1.8,0.0,0.0,...,0.0,5.4,7.2,7.2,7.2,7.2,7.2,7.2,7.2,7.2
4,"1 (Tract 1, Inyo, CA)",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


,geoid,0,1,2,3,4,5,6,7,8,...,14,15,16,17,18,19,20,21,22,23
0,"0 (Tract 9900, Alameda, CA)",11.2,11.2,11.2,11.2,11.2,11.200000,11.2,2.8,0.00,...,911.400001,518.7,61.950000,46.20,46.200,46.2,46.2,46.2,46.2,25.839984
1,"0 (Tract 9901, San Mateo, CA)",0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,236.25,...,0.000000,0.0,0.000000,0.00,0.000,0.0,0.0,0.0,0.0,0.000000
2,"1 (Tract 1, Carson City, NV)",22.4,22.4,22.4,22.4,22.4,10.960008,0.0,0.0,0.00,...,0.000000,0.0,0.000000,0.00,0.000,0.0,0.0,5.6,22.4,22.400000
3,"1 (Tract 1, Fresno, CA)",18.4,18.4,24.0,29.6,29.6,29.600000,24.2,22.4,22.40,...,11.200000,16.6,11.600006,7.20,7.200,7.2,7.2,7.2,7.2,18.400000
4,"1 (Tract 1, Inyo, CA)",0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.00,...,0.000000,0.0,0.000000,68.25,34.125,0.0,0.0,0.0,0.0,0.000000


In [20]:
def get_clean_EV_load_df(year='2025', chargerType='all', scenario="unmanaged", method="mean"):
    EvProjDf = import_ev_projection_df(year, scenario)
    EvProjDf = expand_load_curve_to_hourly(EvProjDf, method=method)
    EvProjDf = sum_ev_loads_by_charger_type(EvProjDf, charger_type=chargerType)
    return EvProjDf
evLoad2035Df = get_clean_EV_load_df(year='2035', chargerType='all', scenario="unmanaged", method="mean")
display(evLoad2035Df.head())

2035


,geoid,0,1,2,3,4,5,6,7,8,...,14,15,16,17,18,19,20,21,22,23
0,"0 (Tract 9900, Alameda, CA)",11.2,11.2,11.2,11.2,11.2,11.200000,11.2,2.8,0.00,...,911.400001,518.7,61.950000,46.20,46.200,46.2,46.2,46.2,46.2,25.839984
1,"0 (Tract 9901, San Mateo, CA)",0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,236.25,...,0.000000,0.0,0.000000,0.00,0.000,0.0,0.0,0.0,0.0,0.000000
2,"1 (Tract 1, Carson City, NV)",22.4,22.4,22.4,22.4,22.4,10.960008,0.0,0.0,0.00,...,0.000000,0.0,0.000000,0.00,0.000,0.0,0.0,5.6,22.4,22.400000
3,"1 (Tract 1, Fresno, CA)",18.4,18.4,24.0,29.6,29.6,29.600000,24.2,22.4,22.40,...,11.200000,16.6,11.600006,7.20,7.200,7.2,7.2,7.2,7.2,18.400000
4,"1 (Tract 1, Inyo, CA)",0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.00,...,0.000000,0.0,0.000000,68.25,34.125,0.0,0.0,0.0,0.0,0.000000


In [19]:
def get_clean_EV_growth_df(year1='2025', chargerType1='all', scenario1="unmanaged", year2='2035', chargerType2="all", scenario2="managed", method="mean"):
    yr1EvProjDf = import_ev_projection_df(year1, scenario1)
    yr1EvProjDf = expand_load_curve_to_hourly(yr1EvProjDf, method=method)
    yr1EvProjDf = sum_ev_loads_by_charger_type(yr1EvProjDf, charger_type=chargerType1)

    yr2EvProjDf = import_ev_projection_df(year2, scenario2)
    yr2EvProjDf = expand_load_curve_to_hourly(yr2EvProjDf, method=method)
    yr2EvProjDf = sum_ev_loads_by_charger_type(yr2EvProjDf, charger_type=chargerType2)

    # # Make a new dataframe and Subtract the hourly values of year 1 from year 2 to get growth
    # growth_df = yr2EvProjDf.copy()
    # hourly_cols = [str(i) for i in range(24)]
    # # growth_df[hourly_cols] = yr2EvProjDf[hourly_cols] - yr1EvProjDf[hourly_cols]

    hourly_cols = [str(i) for i in range(24)]

    # --- 1. Check same geoids ---
    geoids_2025 = set(yr1EvProjDf["geoid"])
    geoids_2035 = set(yr2EvProjDf["geoid"])

    missing_in_2035 = geoids_2025 - geoids_2035
    missing_in_2025 = geoids_2035 - geoids_2025

    if missing_in_2035 or missing_in_2025:
        raise ValueError(
            f"Geoids do not match.\n"
            f"Missing in 2035: {missing_in_2035}\n"
            f"Missing in 2025: {missing_in_2025}"
        )

    # --- 2. Sort & align by geoid ---
    df2025_sorted = yr1EvProjDf.sort_values("geoid").reset_index(drop=True)
    df2035_sorted = yr2EvProjDf.sort_values("geoid").reset_index(drop=True)

    # --- 3. Compute growth (2035 - 2025) ---
    growth_matrix = df2035_sorted[hourly_cols].values - df2025_sorted[hourly_cols].values

    growth_df = pd.DataFrame(growth_matrix, columns=hourly_cols)
    growth_df.insert(0, "geoid", df2025_sorted["geoid"].values)

    return growth_df
    return growth_df
ev_growth_df = get_clean_EV_growth_df(year1='2035', chargerType1='all', scenario1="unmanaged", year2='2035', chargerType2="all", scenario2="unmanaged", method="mean")

2035
2035


## Grid Data

In [ ]:
# LOAD DATA ALL FEEDERS
def import_loadDf_all_feeders(base_path=DATA_DIR):

    relative_folder_path = "cleanedGridData"
    filename = "feeder_load_profile_clean.csv"
    file_path = os.path.join(base_path, relative_folder_path, filename)
    load_df_all_feeders = pd.read_csv(file_path)

    # load_df_all_feeders = pd.read_csv(file_path, dtype={'feeder_id': str})
    return load_df_all_feeders

load_df_all_feeders = import_loadDf_all_feeders()

In [ ]:
# Import metadata geodataframe for all feeders
# 1 row for each feeder
# 'feeder_id'	'division'	'substation'	'nom_volt_kV'	'Existing_DG'	'Queued_DG'	'shape_length'	'geometry'


def import_metaData_gdf_all_feeders(base_path=DATA_DIR):

    relative_folder_path = "cleanedGridData"
    filename = "feeder_meta_clean_gdf.gpkg"
    file_path = os.path.join(base_path, relative_folder_path, filename)
    gdf_all_feeders = gpd.read_file(file_path)
    return gdf_all_feeders


feederMetadataGdf = import_metaData_gdf_all_feeders()
display(feederMetadataGdf.head())

In [ ]:
# Import ICA data from parquet files
def import_feederLineIcaDf_from_parquet(feeder_id, base_path=DATA_DIR):
    """
    get_feeder_ICA_df_from_parquet

    :feeder_id: Feeder ID whose ICA data you want
    :folder_path: Path to the folder in Google Drive that holds the .parquet files
    :return: Pandas DataFrame
    """
    relative_folder_path = "cleanedGridData/ICA_Load_CLEAN_PARQUET_v4"
    filename = f"{feeder_id}.parquet"
    file_path = os.path.join(base_path, relative_folder_path, filename)
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    df = pd.read_parquet(file_path)
    # rint(f"Loaded {filename} with {len(df)} rows and {len(df.columns)} columns.")
    return df

## CENSUS BLOCK DATA FROM THIBAUD

In [ ]:
# Import Census tract data
def import_census_block_gdf_for_charging(base_path=DATA_DIR, unique_geoidStr=None):
    relative_folder_path = "Geography_Files"
    filename = "location_str_to_geoid_mapping.shp"
    file_path = os.path.join(base_path, relative_folder_path, filename)
    gdf = gpd.read_file(file_path)
    if unique_geoidStr is not None:
        gdf = gdf[gdf["GEOID_STR"].isin(unique_geoidStr)]
    return gdf


evCensusGdf = import_census_block_gdf_for_charging()
display(evCensusGdf.head())